# 面试问题：怎样用 Roofline 分析 LLM Prefill、Decode 与容量？

可直接复述的回答：Roofline 用算术强度 FLOPs/Byte 判断算子更接近计算瓶颈还是带宽瓶颈，性能上界是 `min(峰值算力, 带宽×AI)`。Prefill 的大矩阵乘通常复用权重，算术强度较高；单 token decode 反复读取权重，通常更受带宽限制。Batching 能摊薄权重字节，但会增加 KV 和排队。Roofline 是硬件上界，不是延迟预测器，必须乘上实测效率并加入 kernel、通信和调度开销。容量规划还要按 TTFT、TPOT、到达率和副本利用率计算。最后需用真实 trace 校准并保留 headroom。

后续实验使用可读的小型业务数据验证关键判断。所有数值都标记为教学实验，不代表真实 GPU、线上流量或基础模型泛化结果。


## 1. 真实案例：客服推理请求与 GPU 配置输入预览

六条请求覆盖短问答、长文档摘要和多并发聊天。字段包含 prompt、输出 token 和 batch；硬件是教学用 120 TFLOP/s、1.5 TB/s GPU，模型权重 14 GB。


In [1]:
import numpy as np  # 使用 NumPy 计算副本取整与容量指标。
requests05 = [  # 构造六类推理请求配置。
    {"id": "chat-1", "prompt": 128, "output": 64, "batch": 1},  # 单用户短聊天。
    {"id": "chat-8", "prompt": 128, "output": 64, "batch": 8},  # 八路短聊天。
    {"id": "doc-1", "prompt": 4096, "output": 128, "batch": 1},  # 单篇长文档摘要。
    {"id": "doc-4", "prompt": 4096, "output": 128, "batch": 4},  # 四篇长文档批量摘要。
    {"id": "agent-2", "prompt": 1024, "output": 256, "batch": 2},  # 双路工具 Agent。
    {"id": "support-16", "prompt": 256, "output": 96, "batch": 16},  # 高并发客服短答。
]  # 完成具有真实容量字段的请求集。
hardware05 = {"peak_tflops": 120.0, "bandwidth_tb_s": 1.5, "weight_gb": 14.0, "kv_bytes_per_token": 0.5e6}  # 定义教学 GPU 与模型画像。
print("教学实验输入：硬件", hardware05)  # 展示 Roofline 硬件上限。
for request05 in requests05:  # 逐条展示请求长度和并发。
    print(request05)  # 输出一条推理工作负载。


教学实验输入：硬件 {'peak_tflops': 120.0, 'bandwidth_tb_s': 1.5, 'weight_gb': 14.0, 'kv_bytes_per_token': 500000.0}
{'id': 'chat-1', 'prompt': 128, 'output': 64, 'batch': 1}
{'id': 'chat-8', 'prompt': 128, 'output': 64, 'batch': 8}
{'id': 'doc-1', 'prompt': 4096, 'output': 128, 'batch': 1}
{'id': 'doc-4', 'prompt': 4096, 'output': 128, 'batch': 4}
{'id': 'agent-2', 'prompt': 1024, 'output': 256, 'batch': 2}
{'id': 'support-16', 'prompt': 256, 'output': 96, 'batch': 16}


## 2. Baseline（基线）：所有阶段都用峰值算力估时

朴素估算只用 FLOPs/峰值算力，忽略 decode 每步读取权重和 KV，因此会严重低估单 token 延迟。


In [2]:
model_flops_per_token05 = 14e9 * 2.0  # 用两倍参数量近似每 token 前向 FLOPs。
def compute_only_ms05(tokens05, batch05):  # 实现只看峰值算力的延迟下界。
    flops05 = model_flops_per_token05 * tokens05 * batch05  # 估算总浮点运算量。
    return flops05 / (hardware05["peak_tflops"] * 1e12) * 1000.0  # 转换为毫秒计算下界。
baseline_rows05 = []  # 收集只看算力的请求估算。
for request05 in requests05:  # 对每条请求估算完整 token 计算时间。
    total_tokens05 = request05["prompt"] + request05["output"]  # 汇总 prompt 和输出 token。
    estimate05 = compute_only_ms05(total_tokens05, request05["batch"])  # 使用峰值算力估时。
    baseline_rows05.append((request05["id"], round(estimate05, 2)))  # 保存请求与乐观下界。
print("计算峰值基线：request | total_compute_ms")  # 输出基线估算表头。
for row05 in baseline_rows05:  # 逐条展示过度乐观结果。
    print(row05)  # 输出一条计算下界。


计算峰值基线：request | total_compute_ms
('chat-1', 44.8)
('chat-8', 358.4)
('doc-1', 985.6)
('doc-4', 3942.4)
('agent-2', 597.33)
('support-16', 1314.13)


## 3. 核心实现：Prefill/Decode 分阶段 Roofline

Prefill 把权重读取在 prompt token 间摊销；decode 每步只摊到 batch。教学模型同时计算 FLOPs、权重+KV 字节、算术强度、Roofline 性能和阶段下界。


In [3]:
def roofline_phase05(tokens05, batch05, phase05):  # 计算单阶段 Roofline 指标。
    flops05 = model_flops_per_token05 * tokens05 * batch05  # 估算阶段总 FLOPs。
    if phase05 == "prefill":  # 处理 prompt 大矩阵阶段。
        weight_bytes05 = hardware05["weight_gb"] * 1e9  # 一次读取模型权重。
        kv_bytes05 = hardware05["kv_bytes_per_token"] * tokens05 * batch05  # 写入 prompt KV。
    else:  # 处理逐 token decode 阶段。
        weight_bytes05 = hardware05["weight_gb"] * 1e9 * tokens05  # 每个 decode step 读取一次权重。
        kv_bytes05 = hardware05["kv_bytes_per_token"] * tokens05 * batch05 * 2.0  # 近似读写 KV 流量。
    total_bytes05 = weight_bytes05 + kv_bytes05  # 汇总阶段内存流量。
    arithmetic_intensity05 = flops05 / total_bytes05  # 计算 FLOPs/Byte。
    bandwidth_bound05 = hardware05["bandwidth_tb_s"] * 1e12 * arithmetic_intensity05  # 计算带宽决定的 FLOP/s 上界。
    roof_flops05 = min(hardware05["peak_tflops"] * 1e12, bandwidth_bound05)  # 取计算与带宽屋顶较小者。
    latency_ms05 = flops05 / roof_flops05 * 1000.0  # 计算理想 Roofline 延迟下界。
    bottleneck05 = "compute" if roof_flops05 == hardware05["peak_tflops"] * 1e12 else "bandwidth"  # 判断阶段主要屋顶。
    return {"flops": flops05, "bytes": total_bytes05, "ai": arithmetic_intensity05, "roof_tflops": roof_flops05 / 1e12, "latency_ms": latency_ms05, "bottleneck": bottleneck05}  # 返回完整阶段指标。
roofline_rows05 = []  # 收集每个请求的分阶段结果。
for request05 in requests05:  # 对全部工作负载计算 Roofline。
    prefill05 = roofline_phase05(request05["prompt"], request05["batch"], "prefill")  # 计算 prefill 阶段。
    decode05 = roofline_phase05(request05["output"], request05["batch"], "decode")  # 计算 decode 阶段。
    roofline_rows05.append((request05["id"], round(prefill05["ai"], 2), prefill05["bottleneck"], round(prefill05["latency_ms"], 2), round(decode05["ai"], 2), decode05["bottleneck"], round(decode05["latency_ms"], 2)))  # 保存可读阶段对照。
print("核心过程：request | prefill_AI/bound/ms | decode_AI/bound/ms")  # 输出分阶段 Roofline 表头。
for row05 in roofline_rows05:  # 逐请求展示瓶颈差异。
    print(row05)  # 输出一条阶段模型结果。


核心过程：request | prefill_AI/bound/ms | decode_AI/bound/ms
('chat-1', 254.84, 'compute', 29.87, 2.0, 'bandwidth', 597.38)
('chat-8', 1975.74, 'compute', 238.93, 15.99, 'bandwidth', 597.67)
('doc-1', 7146.56, 'compute', 955.73, 2.0, 'bandwidth', 1194.75)
('doc-4', 20671.95, 'compute', 3822.93, 8.0, 'bandwidth', 1195.01)
('agent-2', 3816.83, 'compute', 477.87, 4.0, 'bandwidth', 2389.67)
('support-16', 7146.56, 'compute', 955.73, 31.96, 'bandwidth', 897.02)


## 4. 结果表、实测效率与结果解读

Roofline 下界需要乘以阶段效率才能接近实测。这里假设 prefill 达到屋顶 55%，decode 达到 38%，再计算每请求总延迟和副本容量。


In [4]:
efficiency05 = {"prefill": 0.55, "decode": 0.38}  # 设置教学用阶段实测效率。
capacity_rows05 = []  # 收集校准延迟和每秒请求容量。
for request05 in requests05:  # 对每条请求应用效率校准。
    prefill05 = roofline_phase05(request05["prompt"], request05["batch"], "prefill")  # 重算 prefill 下界。
    decode05 = roofline_phase05(request05["output"], request05["batch"], "decode")  # 重算 decode 下界。
    calibrated_ms05 = prefill05["latency_ms"] / efficiency05["prefill"] + decode05["latency_ms"] / efficiency05["decode"]  # 用实测效率放大理想下界。
    requests_per_second05 = request05["batch"] / (calibrated_ms05 / 1000.0)  # 估算单副本批量吞吐。
    capacity_rows05.append((request05["id"], round(calibrated_ms05, 2), round(requests_per_second05, 2)))  # 保存延迟与容量。
print("request | Roofline校准总延迟ms | 单副本req/s")  # 输出容量结果表头。
for row05 in capacity_rows05:  # 逐工作负载展示校准容量。
    print(row05)  # 输出一条延迟与吞吐估计。
print("结果解读：长prefill提高AI，decode仍反复搬权重；batch越大越能摊销但排队未计入")  # 解释阶段和 batching 机制。


request | Roofline校准总延迟ms | 单副本req/s
('chat-1', 1626.35, 0.61)
('chat-8', 2007.25, 3.99)
('doc-1', 4881.78, 0.2)
('doc-4', 10095.55, 0.4)
('agent-2', 7157.47, 0.28)
('support-16', 4098.29, 3.9)
结果解读：长prefill提高AI，decode仍反复搬权重；batch越大越能摊销但排队未计入


## 5. 失败案例与修正：用理论容量直接定副本数

若流量为 40 req/s，直接按平均吞吐取整会让利用率接近 100%，没有排队和故障余量。修正按 65% 目标利用率并向上取整。


In [5]:
target_request05 = next(row05 for row05 in capacity_rows05 if row05[0] == "chat-8")  # 选取批量客服请求容量。
arrival_rate05 = 40.0  # 设置教学用峰值到达率。
theoretical_capacity05 = target_request05[2]  # 读取单副本理论校准吞吐。
naive_replicas05 = int(np.ceil(arrival_rate05 / theoretical_capacity05))  # 不留 headroom 计算副本数。
safe_replicas05 = int(np.ceil(arrival_rate05 / (theoretical_capacity05 * 0.65)))  # 按65%利用率计算安全副本数。
print("失败行为：理论副本数", naive_replicas05, "预计利用率", round(arrival_rate05 / (naive_replicas05 * theoretical_capacity05), 3))  # 展示过高利用率。
print("修正行为：65%目标利用率副本数", safe_replicas05, "预计利用率", round(arrival_rate05 / (safe_replicas05 * theoretical_capacity05), 3))  # 展示容量 headroom。


失败行为：理论副本数 11 预计利用率 0.911
修正行为：65%目标利用率副本数 16 预计利用率 0.627


## 6. 生产边界与容量制品

真实测量要区分 TTFT、TPOT、输出长度、prefix cache、量化和通信。副本规划还要加入队列模型、流量峰谷、故障域、冷启动和显存约束。


In [6]:
roofline_contract05 = {"gpu_profile": "teaching-120t-1.5tb", "model_weight_gb": hardware05["weight_gb"], "efficiency": efficiency05, "target_utilization": 0.65, "metrics": ["TTFT", "TPOT", "queue_p95"]}  # 定义容量模型版本合同。
print("Roofline容量制品", roofline_contract05)  # 展示硬件、效率和利用率配置。
print("生产替换点：真实profiler、kernel效率、排队模型、输出长度分布、故障headroom和显存门禁")  # 说明上界模型的局限。


Roofline容量制品 {'gpu_profile': 'teaching-120t-1.5tb', 'model_weight_gb': 14.0, 'efficiency': {'prefill': 0.55, 'decode': 0.38}, 'target_utilization': 0.65, 'metrics': ['TTFT', 'TPOT', 'queue_p95']}
生产替换点：真实profiler、kernel效率、排队模型、输出长度分布、故障headroom和显存门禁


## 7. 最小回归测试

断言保护阶段瓶颈、效率校准和容量 headroom。


In [7]:
assert len(requests05) >= 5  # 保证容量案例覆盖多种请求形态。
assert any(row05[2] == "compute" for row05 in roofline_rows05)  # 保证至少一个 prefill 接近计算屋顶。
assert all(row05[5] == "bandwidth" for row05 in roofline_rows05)  # 保证教学 decode 均体现带宽瓶颈。
assert safe_replicas05 >= naive_replicas05  # 保证 headroom 不会减少副本数。
assert roofline_contract05["target_utilization"] < 1.0  # 保证生产合同保留容量余量。
print("最小回归测试通过：分阶段Roofline、效率和副本headroom稳定")  # 显示容量规划关键性质已验证。


最小回归测试通过：分阶段Roofline、效率和副本headroom稳定
